# PaySim Baseline Notebook

This notebook is the stable baseline stage for the HE fraud project.

- Focus on `CASH_OUT` and `TRANSFER`
- Keep all fraud rows
- Downsample non-fraud rows to about a 10:1 negative-to-positive ratio
- Build the 8-dimensional HE-friendly feature vector
- Train two plaintext baselines without relying on `sklearn` model fitting
- Export weights, bias, scaler statistics, and a small HE evaluation slice


In [1]:
from pathlib import Path
import json
import os
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

PROJECT_ROOT = Path('.')
DATA_PATH = PROJECT_ROOT / 'data' / 'PS_20174392719_1491204439457_log.csv'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)

assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH.resolve()}'

SELECTED_TYPES = ['CASH_OUT', 'TRANSFER']
NUMERIC_BASE_FEATURES = ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'deltaOrig', 'deltaDest']
FEATURE_COLUMNS = [
    'amount_scaled',
    'oldbalanceOrg_scaled',
    'oldbalanceDest_scaled',
    'deltaOrig_scaled',
    'deltaDest_scaled',
    'is_cash_out',
    'is_transfer',
    'dest_is_merchant',
]

# These counts were computed from a full chunk scan of the CSV.
TOTAL_ROWS = 6_362_620
TOTAL_FRAUD = 8_213
TOTAL_NON_FRAUD_SELECTED = 2_762_196

NEG_POS_RATIO = 10
TARGET_NEGATIVE_SAMPLES = NEG_POS_RATIO * TOTAL_FRAUD
NEGATIVE_SAMPLE_PROB = TARGET_NEGATIVE_SAMPLES / TOTAL_NON_FRAUD_SELECTED

CHUNK_SIZE = 250_000
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE_WITHIN_TRAIN = 0.2

print('Python executable:', os.environ.get('PYTHONEXECUTABLE', 'not-set'))
print('Dataset:', DATA_PATH.resolve())
print('Target negative samples:', TARGET_NEGATIVE_SAMPLES)
print('Negative sampling probability:', round(NEGATIVE_SAMPLE_PROB, 6))


Python executable: not-set
Dataset: /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/data/PS_20174392719_1491204439457_log.csv
Target negative samples: 82130
Negative sampling probability: 0.029734


In [2]:
overview = pd.DataFrame(
    {
        'metric': [
            'total_rows',
            'total_fraud',
            'fraud_rate',
            'selected_types_non_fraud',
            'target_negative_samples',
        ],
        'value': [
            TOTAL_ROWS,
            TOTAL_FRAUD,
            TOTAL_FRAUD / TOTAL_ROWS,
            TOTAL_NON_FRAUD_SELECTED,
            TARGET_NEGATIVE_SAMPLES,
        ],
    }
)
overview


,metric,value
0,total_rows,"6,362,620.0000"
1,total_fraud,"8,213.0000"
2,fraud_rate,0.0013
3,selected_types_non_fraud,"2,762,196.0000"
4,target_negative_samples,"82,130.0000"


In [3]:
def build_sample_dataframe(data_path: Path, chunk_size: int = CHUNK_SIZE, random_state: int = RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    selected_columns = [
        'step',
        'type',
        'amount',
        'nameOrig',
        'oldbalanceOrg',
        'newbalanceOrig',
        'nameDest',
        'oldbalanceDest',
        'newbalanceDest',
        'isFraud',
        'isFlaggedFraud',
    ]
    sampled_chunks = []
    start_time = time.time()
    total_selected = 0
    total_pos = 0
    total_neg = 0

    for chunk_id, chunk in enumerate(pd.read_csv(data_path, usecols=selected_columns, chunksize=chunk_size)):
        chunk = chunk[chunk['type'].isin(SELECTED_TYPES)].copy()
        if chunk.empty:
            continue

        positives = chunk[chunk['isFraud'] == 1]
        negatives = chunk[chunk['isFraud'] == 0]
        sampled_negatives = negatives.loc[rng.random(len(negatives)) < NEGATIVE_SAMPLE_PROB]

        sampled_chunk = pd.concat([positives, sampled_negatives], ignore_index=True)
        sampled_chunks.append(sampled_chunk)

        total_selected += len(sampled_chunk)
        total_pos += len(positives)
        total_neg += len(sampled_negatives)

        if chunk_id % 5 == 0:
            print(
                f'chunk={chunk_id:02d} elapsed={time.time() - start_time:,.1f}s '
                f'selected={total_selected:,} pos={total_pos:,} neg={total_neg:,}'
            )

    sampled_df = pd.concat(sampled_chunks, ignore_index=True)
    sampled_df = sampled_df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return sampled_df


sample_df = build_sample_dataframe(DATA_PATH)
print('Sample shape:', sample_df.shape)
sample_df['isFraud'].value_counts()


chunk=00 elapsed=0.2s selected=3,350 pos=163 neg=3,187
chunk=05 elapsed=1.2s selected=21,059 pos=1,608 neg=19,451
chunk=10 elapsed=2.2s selected=38,219 pos=2,389 neg=35,830
chunk=15 elapsed=3.2s selected=55,026 pos=3,381 neg=51,645
chunk=20 elapsed=4.3s selected=71,828 pos=4,143 neg=67,685
chunk=25 elapsed=5.2s selected=89,998 pos=8,213 neg=81,785
Sample shape: (89998, 11)


isFraud
0    81785
1     8213
Name: count, dtype: int64

In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['deltaOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
    df['deltaDest'] = df['newbalanceDest'] - df['oldbalanceDest']
    df['is_cash_out'] = (df['type'] == 'CASH_OUT').astype(int)
    df['is_transfer'] = (df['type'] == 'TRANSFER').astype(int)
    df['dest_is_merchant'] = df['nameDest'].astype(str).str.startswith('M').astype(int)
    return df


sample_df = add_features(sample_df)
print(pd.crosstab(sample_df['type'], sample_df['isFraud']))
preview_columns = NUMERIC_BASE_FEATURES + ['is_cash_out', 'is_transfer', 'dest_is_merchant', 'isFraud']
sample_df[preview_columns].head()


isFraud       0     1
type                 
CASH_OUT  65972  4116
TRANSFER  15813  4097


,amount,oldbalanceOrg,oldbalanceDest,deltaOrig,deltaDest,is_cash_out,is_transfer,dest_is_merchant,isFraud
0,"90,430.4000",0.0000,"129,644.3000",0.0000,"90,430.4000",1,0,0,0
1,"98,733.7300",0.0000,"344,035.0000",0.0000,"526,054.5200",1,0,0,0
2,"95,636.2100",367.0000,0.0000,367.0000,"95,636.2100",1,0,0,0
3,"54,294.1700",0.0000,"391,758.4300",0.0000,"54,294.1700",1,0,0,0
4,"481,486.0000","34,518.0000",0.0000,"34,518.0000","481,486.0000",0,1,0,0


In [5]:
def stratified_split_indices(y, test_size=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    y = np.asarray(y)
    train_idx = []
    test_idx = []
    for label in np.unique(y):
        idx = np.flatnonzero(y == label)
        idx = rng.permutation(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.append(idx[:n_test])
        train_idx.append(idx[n_test:])
    train_idx = np.concatenate(train_idx)
    test_idx = np.concatenate(test_idx)
    return rng.permutation(train_idx), rng.permutation(test_idx)


def fit_standard_scaler(X):
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    scale[scale == 0] = 1.0
    return mean, scale


def transform_standard_scaler(X, mean, scale):
    return (X - mean) / scale


def attach_scaled_columns(df, mean, scale):
    df = df.copy()
    transformed = transform_standard_scaler(df[NUMERIC_BASE_FEATURES].to_numpy(dtype=float), mean, scale)
    for i, col in enumerate(NUMERIC_BASE_FEATURES):
        df[f'{col}_scaled'] = transformed[:, i]
    return df


y_all = sample_df['isFraud'].to_numpy()
train_full_idx, test_idx = stratified_split_indices(y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, val_idx = stratified_split_indices(y_all[train_full_idx], test_size=VAL_SIZE_WITHIN_TRAIN, random_state=RANDOM_STATE)

train_full_df = sample_df.iloc[train_full_idx].reset_index(drop=True)
test_df = sample_df.iloc[test_idx].reset_index(drop=True)
train_df = train_full_df.iloc[train_idx].reset_index(drop=True)
val_df = train_full_df.iloc[val_idx].reset_index(drop=True)

mean, scale = fit_standard_scaler(train_df[NUMERIC_BASE_FEATURES].to_numpy(dtype=float))
train_df = attach_scaled_columns(train_df, mean, scale)
val_df = attach_scaled_columns(val_df, mean, scale)
test_df = attach_scaled_columns(test_df, mean, scale)

X_train = train_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_train = train_df['isFraud'].to_numpy(dtype=int)
X_val = val_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_val = val_df['isFraud'].to_numpy(dtype=int)
X_test = test_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_test = test_df['isFraud'].to_numpy(dtype=int)

print('train:', X_train.shape, 'fraud rate:', round(y_train.mean(), 4))
print('val  :', X_val.shape, 'fraud rate:', round(y_val.mean(), 4))
print('test :', X_test.shape, 'fraud rate:', round(y_test.mean(), 4))


train: (57598, 8) fraud rate: 0.0913
val  : (14400, 8) fraud rate: 0.0912
test : (18000, 8) fraud rate: 0.0913


In [6]:
def weighted_standardize(X):
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    scale[scale == 0] = 1.0
    return (X - mean) / scale, mean, scale


def fit_ridge_score_model(X, y_pm1, alpha=1.0):
    Xz, x_mean, x_scale = weighted_standardize(X)
    yz = y_pm1.astype(float)
    X_aug = np.column_stack([Xz, np.ones(len(Xz))])
    reg = alpha * np.eye(X_aug.shape[1])
    reg[-1, -1] = 0.0
    coef = np.linalg.solve(X_aug.T @ X_aug + reg, X_aug.T @ yz)

    weights_z = coef[:-1]
    bias_z = coef[-1]

    weights_raw = weights_z / x_scale
    bias_raw = bias_z - np.sum((weights_z * x_mean) / x_scale)
    return weights_raw, bias_raw


def fit_logistic_regression(X, y, l2=1.0):
    X_aug = np.column_stack([X, np.ones(len(X))])
    y = y.astype(float)

    def objective(w):
        z = X_aug @ w
        p = expit(z)
        eps = 1e-12
        loss = -(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps)).mean()
        reg = 0.5 * l2 * np.sum(w[:-1] ** 2) / len(y)
        return loss + reg

    def gradient(w):
        z = X_aug @ w
        p = expit(z)
        grad = (X_aug.T @ (p - y)) / len(y)
        grad[:-1] += l2 * w[:-1] / len(y)
        return grad

    init = np.zeros(X_aug.shape[1], dtype=float)
    result = minimize(objective, init, jac=gradient, method='L-BFGS-B', options={'maxiter': 500})
    return result.x[:-1], result.x[-1], result


def confusion_counts(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tn, fp, fn, tp


def average_precision(y_true, scores):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    tp_cum = np.cumsum(y_sorted == 1)
    fp_cum = np.cumsum(y_sorted == 0)
    precision = tp_cum / (tp_cum + fp_cum)
    total_pos = max(int((y_true == 1).sum()), 1)
    return float((precision[y_sorted == 1]).sum() / total_pos)


def evaluate_predictions(y_true, y_pred, y_score, model_name):
    tn, fp, fn, tp = confusion_counts(y_true, y_pred)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {
        'model': model_name,
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'pr_auc': average_precision(y_true, y_score),
        'confusion_matrix': [[tn, fp], [fn, tp]],
        'predicted_positive': int(y_pred.sum()),
    }


In [7]:
ridge_w, ridge_b = fit_ridge_score_model(X_train, 2 * y_train - 1, alpha=1.0)
ridge_val_score = X_val @ ridge_w + ridge_b
ridge_val_pred = (ridge_val_score > 0).astype(int)
ridge_test_score = X_test @ ridge_w + ridge_b
ridge_test_pred = (ridge_test_score > 0).astype(int)

logreg_w, logreg_b, logreg_result = fit_logistic_regression(X_train, y_train, l2=1.0)
logreg_val_logit = X_val @ logreg_w + logreg_b
logreg_val_prob = expit(logreg_val_logit)
logreg_val_pred = (logreg_val_prob >= 0.5).astype(int)
logreg_test_logit = X_test @ logreg_w + logreg_b
logreg_test_prob = expit(logreg_test_logit)
logreg_test_pred = (logreg_test_prob >= 0.5).astype(int)

metrics_df = pd.DataFrame(
    [
        evaluate_predictions(y_val, ridge_val_pred, ridge_val_score, 'ridge_score_val'),
        evaluate_predictions(y_test, ridge_test_pred, ridge_test_score, 'ridge_score_test'),
        evaluate_predictions(y_val, logreg_val_pred, logreg_val_prob, 'logreg_val'),
        evaluate_predictions(y_test, logreg_test_pred, logreg_test_prob, 'logreg_test'),
    ]
)
metrics_df


,model,precision,recall,f1,pr_auc,confusion_matrix,predicted_positive
0,ridge_score_val,1.0000,0.1963,0.3282,0.7522,"[[13086, 0], [1056, 258]]",258
1,ridge_score_test,1.0000,0.1923,0.3226,0.7414,"[[16357, 0], [1327, 316]]",316
2,logreg_val,0.9388,0.7002,0.8021,0.9048,"[[13026, 60], [394, 920]]",980
3,logreg_test,0.9282,0.6847,0.7881,0.8989,"[[16270, 87], [518, 1125]]",1212


In [8]:
feature_order = {
    'feature_columns': FEATURE_COLUMNS,
    'numeric_base_features': NUMERIC_BASE_FEATURES,
    'binary_features': ['is_cash_out', 'is_transfer', 'dest_is_merchant'],
}

artifacts = {
    'feature_order': feature_order,
    'scaler': {
        'mean': mean.tolist(),
        'scale': scale.tolist(),
    },
    'ridge_score': {
        'weights': ridge_w.tolist(),
        'bias': float(ridge_b),
    },
    'logistic_regression': {
        'weights': logreg_w.tolist(),
        'bias': float(logreg_b),
        'optimizer_success': bool(logreg_result.success),
        'optimizer_message': str(logreg_result.message),
    },
    'sample_summary': {
        'rows': int(len(sample_df)),
        'fraud': int(sample_df['isFraud'].sum()),
        'non_fraud': int((sample_df['isFraud'] == 0).sum()),
        'types': {str(k): int(v) for k, v in sample_df['type'].value_counts().to_dict().items()},
    },
    'metrics': metrics_df.to_dict(orient='records'),
}

artifact_path = ARTIFACT_DIR / 'baseline_model_artifacts.json'
artifact_path.write_text(json.dumps(artifacts, indent=2))

def make_he_eval_subset(test_meta_df, scaled_feature_df, n_pos=32, n_neg=96, random_state=42):
    rng = np.random.default_rng(random_state)
    test_meta_df = test_meta_df.reset_index(drop=True)
    scaled_feature_df = scaled_feature_df.reset_index(drop=True)

    pos_idx = test_meta_df.index[test_meta_df['isFraud'] == 1].to_numpy()
    neg_idx = test_meta_df.index[test_meta_df['isFraud'] == 0].to_numpy()

    pos_take = min(n_pos, len(pos_idx))
    neg_take = min(n_neg, len(neg_idx))

    pos_sel = rng.choice(pos_idx, size=pos_take, replace=False) if pos_take else np.array([], dtype=int)
    neg_sel = rng.choice(neg_idx, size=neg_take, replace=False) if neg_take else np.array([], dtype=int)

    selected = np.concatenate([pos_sel, neg_sel])
    rng.shuffle(selected)
    return test_meta_df.loc[selected].reset_index(drop=True), scaled_feature_df.loc[selected].reset_index(drop=True)


test_feature_df = test_df[FEATURE_COLUMNS].copy()
he_meta_df, he_X_df = make_he_eval_subset(test_df, test_feature_df, n_pos=32, n_neg=96, random_state=RANDOM_STATE)
he_meta_df.to_csv(ARTIFACT_DIR / 'he_eval_meta.csv', index=False)
he_X_df.to_csv(ARTIFACT_DIR / 'he_eval_scaled_features.csv', index=False)

print('Saved:', artifact_path.resolve())
print('HE eval subset:', he_X_df.shape)
he_meta_df['isFraud'].value_counts()


Saved: /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/baseline_model_artifacts.json
HE eval subset: (128, 8)


isFraud
0    96
1    32
Name: count, dtype: int64

## Notes

- This notebook avoids `sklearn` model fitting because the current local environment hit an OpenMP shared-memory error during training.
- The logistic regression artifact is still in the exact form needed later for HE-friendly inference: `score = w^T x + b`.
- When you are ready for HE, use the exported scaled features and compare encrypted raw score against plaintext raw score first.
